<a href="https://colab.research.google.com/github/osmarbraz/sri/blob/main/0_0_PrepararDocumentosCSV_trainjson_v1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"></a>

# Preparacao do arquivo documentos.csv a partir de train.json

Este notebook le `data/train.json`, extrai o campo `title`, gera `data/documentos.csv` com as colunas `id` e `documento`, limita a saida a 500 registros e garante no maximo 512 tokens por titulo.

No Jupyter, a instalacao deve ser feita com `%pip install transformers==4.49.0 -q`, porque isso usa o ambiente do kernel ativo. No terminal do Windows, o equivalente mais confiavel e `py -m pip install transformers==4.49.0 -q`.

In [5]:
%pip install transformers==4.49.0 -q

Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
import json
from pathlib import Path

import pandas as pd
from transformers import AutoTokenizer

INPUT_PATH = Path('data') / 'train.json'
OUTPUT_PATH = Path('data') / 'documentos.csv'
MODEL_NAME = 'neuralmind/bert-base-portuguese-cased'
MAX_TOKENS = 512
MAX_ROWS = 500

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
print(f'Tokenizer carregado: {MODEL_NAME}')

c:\Users\Fernando Paladini\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
None of PyTorch, TensorFlow >= 2.0, or Flax have been found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


Tokenizer carregado: neuralmind/bert-base-portuguese-cased


In [7]:
def normalize_text(value):
    if value is None:
        return ''
    if not isinstance(value, str):
        value = str(value)
    return ' '.join(value.split()).strip()


def truncate_to_max_tokens(text, max_tokens=MAX_TOKENS):
    token_ids = tokenizer.encode(text, add_special_tokens=False)
    if len(token_ids) > max_tokens:
        token_ids = token_ids[:max_tokens]
    return tokenizer.decode(token_ids, skip_special_tokens=True, clean_up_tokenization_spaces=True).strip()


with INPUT_PATH.open('r', encoding='utf-8') as f:
    data = json.load(f)

if isinstance(data, dict):
    if 'train' in data and isinstance(data['train'], list):
        records = data['train']
    else:
        records = list(data.values())
elif isinstance(data, list):
    records = data
else:
    raise ValueError('Formato de train.json nao reconhecido.')

rows = []
for record in records:
    if len(rows) >= MAX_ROWS:
        break

    title = normalize_text(record.get('title')) if isinstance(record, dict) else ''
    if not title:
        continue

    title = truncate_to_max_tokens(title)
    rows.append({'id': len(rows) + 1, 'documento': title})

df = pd.DataFrame(rows, columns=['id', 'documento'])
df.head()

,id,documento
0,1,Brasil inicia construção do 5 Plano de Ação Na...
1,2,CGU apoia evento para criação de laboratórios ...
2,3,Covid - 19 CGU e PF apuram irregularidades na ...
3,4,Covid - 19 CGU e PF aprofundam investigações d...
4,5,CGU convida cidadãos a participarem do seu Con...


In [8]:
df.to_csv(OUTPUT_PATH, index=False, encoding='utf-8', sep=';')

print(f'Arquivo gerado: {OUTPUT_PATH}')
print(f'Quantidade de linhas: {len(df)}')
print(df.head(3).to_string(index=False))

Arquivo gerado: data\documentos.csv
Quantidade de linhas: 500
 id                                                                  documento
  1     Brasil inicia construção do 5 Plano de Ação Nacional de Governo Aberto
  2 CGU apoia evento para criação de laboratórios de inovação no setor público
  3 Covid - 19 CGU e PF apuram irregularidades na Secretaria de Saúde do Piauí


In [9]:
assert len(df) <= MAX_ROWS
assert all(len(tokenizer.encode(text, add_special_tokens=False)) <= MAX_TOKENS for text in df['documento'])
print('Validacao concluida: no maximo 500 linhas e no maximo 512 tokens por documento.')

Validacao concluida: no maximo 500 linhas e no maximo 512 tokens por documento.
